# Reto práctico: informe ejecutivo de ventas con PostgreSQL

## Contexto del problema

Una empresa registra sus ventas por año, trimestre, región y producto. El equipo de dirección necesita un informe compacto para comparar el rendimiento y preparar una vista para reporting.

Los datos son sintéticos y están incluidos en este notebook. El reto consiste en construir cuatro vistas relacionadas con el mismo conjunto de datos.

> **Entorno:** PostgreSQL. `CROSSTAB` requiere la extensión `tablefunc`.

## Datos disponibles

La tabla `sales_data` contiene 96 registros sintéticos de 2024 y 2025, cuatro regiones y tres productos.

- `year`: año.
- `quarter`: trimestre (`Q1` a `Q4`).
- `region`: región comercial.
- `product`: producto.
- `units`: unidades vendidas.
- `sales`: importe de ventas.

La siguiente celda prepara la tabla. No forma parte de la resolución.

In [ ]:
# Preparación del entorno SQL.
# Requiere PostgreSQL y una conexión accesible desde el notebook.
# Define DATABASE_URL en el entorno antes de ejecutar las celdas SQL.

import os

try:
    import psycopg
    _psycopg_api = "psycopg"
except ImportError:
    try:
        import psycopg2
        _psycopg_api = "psycopg2"
    except ImportError as exc:
        raise ImportError(
            "Este notebook necesita psycopg o psycopg2 para conectarse a PostgreSQL."
        ) from exc

DATABASE_URL = os.environ.get("DATABASE_URL")
if not DATABASE_URL:
    raise RuntimeError("Define DATABASE_URL antes de ejecutar las celdas SQL.")

if _psycopg_api == "psycopg":
    conn = psycopg.connect(DATABASE_URL)
else:
    conn = psycopg2.connect(DATABASE_URL)

def run_sql(sql, params=None, show=True):
    cur = conn.cursor()
    cur.execute(sql, params)
    if cur.description:
        columns = [d[0] for d in cur.description]
        rows = cur.fetchall()
        if show:
            from pprint import pprint
            pprint([dict(zip(columns, row)) for row in rows])
        cur.close()
        return columns, rows
    conn.commit()
    cur.close()
    return None, None

run_sql("DROP TABLE IF EXISTS sales_data;")

run_sql("""
CREATE TABLE sales_data (
    year INTEGER NOT NULL,
    quarter VARCHAR(2) NOT NULL,
    region VARCHAR(20) NOT NULL,
    product VARCHAR(20) NOT NULL,
    units INTEGER NOT NULL,
    sales INTEGER NOT NULL
);
""")

run_sql("""
INSERT INTO sales_data (year, quarter, region, product, units, sales) VALUES

(2024, 'Q1', 'North', 'Core', 134, 5360),
(2024, 'Q1', 'North', 'Plus', 91, 5005),
(2024, 'Q1', 'North', 'Pro', 62, 4960),
(2024, 'Q1', 'South', 'Core', 121, 4840),
(2024, 'Q1', 'South', 'Plus', 83, 4565),
(2024, 'Q1', 'South', 'Pro', 59, 4720),
(2024, 'Q1', 'East', 'Core', 118, 4720),
(2024, 'Q1', 'East', 'Plus', 90, 4950),
(2024, 'Q1', 'East', 'Pro', 83, 6640),
(2024, 'Q1', 'West', 'Core', 128, 5120),
(2024, 'Q1', 'West', 'Plus', 118, 6490),
(2024, 'Q1', 'West', 'Pro', 91, 7280),
(2024, 'Q2', 'North', 'Core', 139, 5560),
(2024, 'Q2', 'North', 'Plus', 96, 5280),
(2024, 'Q2', 'North', 'Pro', 84, 6720),
(2024, 'Q2', 'South', 'Core', 115, 4600),
(2024, 'Q2', 'South', 'Plus', 79, 4345),
(2024, 'Q2', 'South', 'Pro', 54, 4320),
(2024, 'Q2', 'East', 'Core', 126, 5040),
(2024, 'Q2', 'East', 'Plus', 102, 5610),
(2024, 'Q2', 'East', 'Pro', 74, 5920),
(2024, 'Q2', 'West', 'Core', 124, 4960),
(2024, 'Q2', 'West', 'Plus', 102, 5610),
(2024, 'Q2', 'West', 'Pro', 58, 4640),
(2024, 'Q3', 'North', 'Core', 123, 4920),
(2024, 'Q3', 'North', 'Plus', 87, 4785),
(2024, 'Q3', 'North', 'Pro', 79, 6320),
(2024, 'Q3', 'South', 'Core', 138, 5520),
(2024, 'Q3', 'South', 'Plus', 113, 6215),
(2024, 'Q3', 'South', 'Pro', 81, 6480),
(2024, 'Q3', 'East', 'Core', 130, 5200),
(2024, 'Q3', 'East', 'Plus', 97, 5335),
(2024, 'Q3', 'East', 'Pro', 77, 6160),
(2024, 'Q3', 'West', 'Core', 141, 5640),
(2024, 'Q3', 'West', 'Plus', 103, 5665),
(2024, 'Q3', 'West', 'Pro', 67, 5360),
(2024, 'Q4', 'North', 'Core', 144, 5760),
(2024, 'Q4', 'North', 'Plus', 98, 5390),
(2024, 'Q4', 'North', 'Pro', 87, 6960),
(2024, 'Q4', 'South', 'Core', 106, 4240),
(2024, 'Q4', 'South', 'Plus', 81, 4455),
(2024, 'Q4', 'South', 'Pro', 57, 4560),
(2024, 'Q4', 'East', 'Core', 138, 5520),
(2024, 'Q4', 'East', 'Plus', 110, 6050),
(2024, 'Q4', 'East', 'Pro', 97, 7760),
(2024, 'Q4', 'West', 'Core', 120, 4800),
(2024, 'Q4', 'West', 'Plus', 88, 4840),
(2024, 'Q4', 'West', 'Pro', 61, 4880),
(2025, 'Q1', 'North', 'Core', 136, 5440),
(2025, 'Q1', 'North', 'Plus', 99, 5445),
(2025, 'Q1', 'North', 'Pro', 78, 6240),
(2025, 'Q1', 'South', 'Core', 118, 4720),
(2025, 'Q1', 'South', 'Plus', 101, 5555),
(2025, 'Q1', 'South', 'Pro', 66, 5280),
(2025, 'Q1', 'East', 'Core', 122, 4880),
(2025, 'Q1', 'East', 'Plus', 117, 6435),
(2025, 'Q1', 'East', 'Pro', 80, 6400),
(2025, 'Q1', 'West', 'Core', 153, 6120),
(2025, 'Q1', 'West', 'Plus', 108, 5940),
(2025, 'Q1', 'West', 'Pro', 86, 6880),
(2025, 'Q2', 'North', 'Core', 135, 5400),
(2025, 'Q2', 'North', 'Plus', 120, 6600),
(2025, 'Q2', 'North', 'Pro', 82, 6560),
(2025, 'Q2', 'South', 'Core', 131, 5240),
(2025, 'Q2', 'South', 'Plus', 104, 5720),
(2025, 'Q2', 'South', 'Pro', 71, 5680),
(2025, 'Q2', 'East', 'Core', 153, 6120),
(2025, 'Q2', 'East', 'Plus', 110, 6050),
(2025, 'Q2', 'East', 'Pro', 96, 7680),
(2025, 'Q2', 'West', 'Core', 120, 4800),
(2025, 'Q2', 'West', 'Plus', 92, 5060),
(2025, 'Q2', 'West', 'Pro', 85, 6800),
(2025, 'Q3', 'North', 'Core', 122, 4880),
(2025, 'Q3', 'North', 'Plus', 113, 6215),
(2025, 'Q3', 'North', 'Pro', 71, 5680),
(2025, 'Q3', 'South', 'Core', 131, 5240),
(2025, 'Q3', 'South', 'Plus', 106, 5830),
(2025, 'Q3', 'South', 'Pro', 73, 5840),
(2025, 'Q3', 'East', 'Core', 139, 5560),
(2025, 'Q3', 'East', 'Plus', 106, 5830),
(2025, 'Q3', 'East', 'Pro', 83, 6640),
(2025, 'Q3', 'West', 'Core', 153, 6120),
(2025, 'Q3', 'West', 'Plus', 114, 6270),
(2025, 'Q3', 'West', 'Pro', 78, 6240),
(2025, 'Q4', 'North', 'Core', 142, 5680),
(2025, 'Q4', 'North', 'Plus', 112, 6160),
(2025, 'Q4', 'North', 'Pro', 77, 6160),
(2025, 'Q4', 'South', 'Core', 122, 4880),
(2025, 'Q4', 'South', 'Plus', 86, 4730),
(2025, 'Q4', 'South', 'Pro', 76, 6080),
(2025, 'Q4', 'East', 'Core', 167, 6680),
(2025, 'Q4', 'East', 'Plus', 133, 7315),
(2025, 'Q4', 'East', 'Pro', 82, 6560),
(2025, 'Q4', 'West', 'Core', 139, 5560),
(2025, 'Q4', 'West', 'Plus', 113, 6215),
(2025, 'Q4', 'West', 'Pro', 70, 5600)
;
""")

run_sql("SELECT COUNT(*) AS rows_loaded FROM sales_data;")


## Objetivo

Construir cuatro vistas del mismo informe:

1. Clasificación trimestral de las regiones de 2025 en formato horizontal.
2. Totales jerárquicos por año y trimestre con `ROLLUP`.
3. Totales por región y producto, incluyendo todas las combinaciones de agrupación con `CUBE`.
4. Las dos regiones con mayor volumen de ventas de cada año en un único texto con `STRING_AGG`.

La clasificación y el resumen anual deben ser deterministas incluso cuando haya empates.

## Criterios de aceptación

- La clasificación debe conservar las cuatro regiones y los cuatro trimestres de 2025.
- Cada trimestre debe tener posiciones 1–4, usando un desempate determinista por región.
- `ROLLUP` debe mostrar detalle trimestral, subtotal anual y total general, con etiquetas legibles.
- `CUBE` debe mostrar detalle, totales por región, totales por producto y total general.
- El resumen anual debe contener exactamente dos regiones por año.
- Todo debe ser SQL ejecutable en PostgreSQL y depender únicamente de `sales_data`.

# Solución — consultar después de intentar

Las cinco exigencias se resuelven en **cuatro vistas**. Se usan funciones de ventana, `CROSSTAB`, `ROLLUP`, `CUBE`, `COALESCE` y `STRING_AGG`.

### 1. Clasificación trimestral y tabla horizontal

In [ ]:
pivot_query = """
CREATE EXTENSION IF NOT EXISTS tablefunc;

SELECT *
FROM CROSSTAB($$
    WITH regional_quarter_sales AS (
        SELECT
            region,
            quarter,
            SUM(sales) AS total_sales
        FROM sales_data
        WHERE year = 2025
        GROUP BY region, quarter
    )
    SELECT
        region,
        quarter,
        ROW_NUMBER() OVER (
            PARTITION BY quarter
            ORDER BY total_sales DESC, region ASC
        )::INTEGER AS position
    FROM regional_quarter_sales
    ORDER BY region ASC, quarter ASC
$$) AS ct (
    region VARCHAR,
    "Q1" INTEGER,
    "Q2" INTEGER,
    "Q3" INTEGER,
    "Q4" INTEGER
)
ORDER BY region ASC;
"""
run_sql(pivot_query)


### 2. Totales jerárquicos por año y trimestre

In [ ]:
rollup_query = """
SELECT
    COALESCE(year::TEXT, 'All years') AS year,
    COALESCE(quarter, 'All quarters') AS quarter,
    SUM(sales) AS total_sales
FROM sales_data
GROUP BY ROLLUP(year, quarter)
ORDER BY
    CASE WHEN year IS NULL THEN 1 ELSE 0 END,
    year ASC,
    CASE WHEN quarter IS NULL THEN 1 ELSE 0 END,
    quarter ASC;
"""
run_sql(rollup_query)


### 3. Totales por región, producto y todas sus combinaciones

In [ ]:
cube_query = """
SELECT
    COALESCE(region, 'All regions') AS region,
    COALESCE(product, 'All products') AS product,
    SUM(sales) AS total_sales
FROM sales_data
GROUP BY CUBE(region, product)
ORDER BY
    GROUPING(region),
    region ASC,
    GROUPING(product),
    product ASC;
"""
run_sql(cube_query)


### 4. Dos regiones principales por año en un solo valor

In [ ]:
string_agg_query = """
WITH region_totals AS (
    SELECT
        year,
        region,
        SUM(sales) AS total_sales
    FROM sales_data
    GROUP BY year, region
),
ranked_regions AS (
    SELECT
        year,
        region,
        ROW_NUMBER() OVER (
            PARTITION BY year
            ORDER BY total_sales DESC, region ASC
        )::INTEGER AS position
    FROM region_totals
)
SELECT
    year,
    STRING_AGG(region, ', ' ORDER BY position) AS top_regions
FROM ranked_regions
WHERE position <= 2
GROUP BY year
ORDER BY year ASC;
"""
run_sql(string_agg_query)


## Comprobaciones rápidas

- La primera vista debe tener 4 filas y 5 columnas.
- `ROLLUP` debe incluir 8 filas trimestrales, 2 subtotales anuales y 1 total general.
- `CUBE` debe incluir detalle región-producto, subtotales por cada dimensión y el total general.
- El resumen final debe contener una fila por año y exactamente dos regiones en cada valor.

### Por qué se usan `ROW_NUMBER()` y un desempate

`RANK()` puede asignar el mismo puesto a varias regiones y, al filtrar `rank <= 2`, devolver más de dos regiones. Aquí el requisito pide exactamente dos. `ROW_NUMBER()` garantiza posiciones únicas; `region ASC` hace que el resultado sea reproducible si las ventas empatan.

## Interpretación y autoevaluación

- La clasificación se reinicia dentro de cada trimestre porque `quarter` es la partición de la ventana.
- `ROLLUP(year, quarter)` representa la jerarquía año → trimestre y genera subtotales y total general.
- `CUBE(region, product)` genera todos los niveles de agregación posibles de esas dos dimensiones.
- `COALESCE` convierte los `NULL` de las filas de totales en etiquetas comprensibles.
- `STRING_AGG` permite presentar las dos regiones principales de cada año como un único valor de texto.
- El desempate explícito evita resultados ambiguos y garantiza exactamente dos regiones en el resumen anual.

### Autoevaluación

- [ ] Puedo explicar por qué la clasificación se reinicia en cada trimestre.
- [ ] Puedo distinguir `ROLLUP` de `CUBE`.
- [ ] Puedo explicar por qué `ROW_NUMBER()` es adecuado cuando se necesitan exactamente dos filas.
- [ ] Puedo interpretar los `NULL` producidos por las filas de totales.
- [ ] Puedo explicar cuándo resulta útil `STRING_AGG`.